# 08 — Official MS-TCN Full Breakfast Run — Multi-cell Version

This is the same full official MS-TCN baseline pipeline, but split into clear cells.

Run order:

1. Run cells from top to bottom.
2. The training cell saves checkpoints after every epoch directly to Google Drive.
3. If the runtime dies, re-run the notebook; the training cell can resume from the latest saved epoch.

This notebook trains a **visual-only official MS-TCN baseline** on Breakfast split 1 for 30 epochs.

## 1. Mount Drive and imports

In [7]:

# Official MS-TCN full Breakfast run — one-shot Colab pipeline


from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os
import re
import sys
import csv
import json
import time
import shutil
import random
import subprocess
import py_compile
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch import optim
from tqdm.auto import tqdm

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 0. User configuration

In [8]:
DATASET = "breakfast"
SPLIT = 1

EPOCHS = 30

FORCE_RETRAIN = False

REQUIRE_GPU = False

# Official MS-TCN defaults.
NUM_STAGES = 4
NUM_LAYERS = 10
NUM_F_MAPS = 64
FEATURE_DIM = 2048
BATCH_SIZE = 1
LEARNING_RATE = 5e-4
SAMPLE_RATE = 1

SEED = 1538574472

DRIVE_ROOT = Path("/content/drive/MyDrive/mmf_tas_lab_data")

BREAKFAST_DRIVE_ROOT = DRIVE_ROOT / "zenodo_ms_tcn_data" / "breakfast"
FEATURE_DIR_DRIVE = BREAKFAST_DRIVE_ROOT / "features"
GT_DIR_DRIVE = BREAKFAST_DRIVE_ROOT / "groundTruth"
SPLIT_DIR_DRIVE = BREAKFAST_DRIVE_ROOT / "splits"
MAPPING_PATH_DRIVE = BREAKFAST_DRIVE_ROOT / "mapping.txt"

LOCAL_BREAKFAST_ROOT = Path("/content/breakfast_local")
LOCAL_FEATURE_DIR = LOCAL_BREAKFAST_ROOT / "features"
LOCAL_GT_DIR = LOCAL_BREAKFAST_ROOT / "groundTruth"
LOCAL_SPLIT_DIR = LOCAL_BREAKFAST_ROOT / "splits"
LOCAL_MAPPING_PATH = LOCAL_BREAKFAST_ROOT / "mapping.txt"

REPO_ROOT = Path("/content/ms-tcn-official")
WORK_ROOT = Path("/content/ms-tcn-official-working")

RUN_NAME = f"official_full_e{EPOCHS}"
ARTIFACT_ROOT = DRIVE_ROOT / "text_assisted_tas" / "breakfast" / f"official_mstcn_full_e{EPOCHS}_oneshot"

MODEL_DIR = ARTIFACT_ROOT / "models" / DATASET / f"split_{SPLIT}_{RUN_NAME}"
RESULTS_DIR = ARTIFACT_ROOT / "results" / DATASET / f"split_{SPLIT}_{RUN_NAME}"
EVAL_DIR = ARTIFACT_ROOT / "evaluation"
LOG_DIR = ARTIFACT_ROOT / "logs"

for p in [MODEL_DIR, RESULTS_DIR, EVAL_DIR, LOG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("========== Configuration ==========")
print("DATASET:", DATASET)
print("SPLIT:", SPLIT)
print("EPOCHS:", EPOCHS)
print("RUN_NAME:", RUN_NAME)
print("ARTIFACT_ROOT:", ARTIFACT_ROOT)
print("MODEL_DIR:", MODEL_DIR)
print("RESULTS_DIR:", RESULTS_DIR)

========== Configuration ==========
DATASET: breakfast
SPLIT: 1
EPOCHS: 30
RUN_NAME: official_full_e30
ARTIFACT_ROOT: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/official_mstcn_full_e30_oneshot
MODEL_DIR: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/official_mstcn_full_e30_oneshot/models/breakfast/split_1_official_full_e30
RESULTS_DIR: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/official_mstcn_full_e30_oneshot/results/breakfast/split_1_official_full_e30


## 1. GPU check

In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("\n========== Device ==========")
print("device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU: unavailable")

if REQUIRE_GPU and device.type != "cuda":
    raise RuntimeError(
        "GPU is required for the full official MS-TCN run. "
        "Set Runtime → Change runtime type → GPU, then run this notebook again."
    )


========== Device ==========
device: cpu
GPU: unavailable


## 2. Seed

In [10]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

try:
    torch.backends.cudnn.deterministic = True
except Exception:
    pass

## 3. Validate Drive dataset

In [11]:
print("\n========== Validate Drive dataset ==========")

required_paths = [
    BREAKFAST_DRIVE_ROOT,
    FEATURE_DIR_DRIVE,
    GT_DIR_DRIVE,
    SPLIT_DIR_DRIVE,
    MAPPING_PATH_DRIVE,
]

for p in required_paths:
    print(p, "exists:", p.exists())
    assert p.exists(), p

train_split_drive = SPLIT_DIR_DRIVE / f"train.split{SPLIT}.bundle"
test_split_drive = SPLIT_DIR_DRIVE / f"test.split{SPLIT}.bundle"

assert train_split_drive.exists(), train_split_drive
assert test_split_drive.exists(), test_split_drive

train_entries_drive = [x.strip() for x in train_split_drive.read_text().splitlines() if x.strip()]
test_entries_drive = [x.strip() for x in test_split_drive.read_text().splitlines() if x.strip()]

print("Drive features:", len(list(FEATURE_DIR_DRIVE.glob("*.npy"))))
print("Drive groundTruth:", len(list(GT_DIR_DRIVE.glob("*.txt"))))
print("Drive splits:", len(list(SPLIT_DIR_DRIVE.glob("*"))))
print("train entries:", len(train_entries_drive))
print("test entries:", len(test_entries_drive))


========== Validate Drive dataset ==========
/content/drive/MyDrive/mmf_tas_lab_data/zenodo_ms_tcn_data/breakfast exists: True
/content/drive/MyDrive/mmf_tas_lab_data/zenodo_ms_tcn_data/breakfast/features exists: True
/content/drive/MyDrive/mmf_tas_lab_data/zenodo_ms_tcn_data/breakfast/groundTruth exists: True
/content/drive/MyDrive/mmf_tas_lab_data/zenodo_ms_tcn_data/breakfast/splits exists: True
/content/drive/MyDrive/mmf_tas_lab_data/zenodo_ms_tcn_data/breakfast/mapping.txt exists: True
Drive features: 1712
Drive groundTruth: 1712
Drive splits: 9
train entries: 1460
test entries: 252


## 4. Copy Breakfast to local Colab disk

In [12]:
print("\n========== Copy dataset to local Colab disk ==========")

def local_dataset_is_complete():
    if not LOCAL_BREAKFAST_ROOT.exists():
        return False
    if len(list(LOCAL_FEATURE_DIR.glob("*.npy"))) != 1712:
        return False
    if len(list(LOCAL_GT_DIR.glob("*.txt"))) != 1712:
        return False
    if not LOCAL_MAPPING_PATH.exists():
        return False

    sample_video = Path(train_entries_drive[0]).stem
    if not (LOCAL_FEATURE_DIR / f"{sample_video}.npy").exists():
        return False
    if not (LOCAL_GT_DIR / f"{sample_video}.txt").exists():
        return False

    return True


def copy_dir_with_rsync_or_shutil(src, dst):
    src = Path(src)
    dst = Path(dst)

    if dst.exists():
        shutil.rmtree(dst)

    dst.parent.mkdir(parents=True, exist_ok=True)

    rsync_cmd = ["rsync", "-ah", "--info=progress2", f"{src}/", f"{dst}/"]
    try:
        print("Running:", " ".join(rsync_cmd))
        subprocess.run(rsync_cmd, check=True)
    except Exception as e:
        print("rsync failed, falling back to shutil.copytree")
        print("Reason:", repr(e))
        shutil.copytree(src, dst)


if FORCE_RETRAIN:
    print("FORCE_RETRAIN=True")
    print("This does not delete local data, but training checkpoints may be overwritten.")

if local_dataset_is_complete():
    print("Local dataset already complete. Skipping copy:", LOCAL_BREAKFAST_ROOT)
else:
    if LOCAL_BREAKFAST_ROOT.exists():
        print("Removing incomplete local copy:", LOCAL_BREAKFAST_ROOT)
        shutil.rmtree(LOCAL_BREAKFAST_ROOT)

    LOCAL_BREAKFAST_ROOT.mkdir(parents=True, exist_ok=True)

    copy_dir_with_rsync_or_shutil(FEATURE_DIR_DRIVE, LOCAL_FEATURE_DIR)
    copy_dir_with_rsync_or_shutil(GT_DIR_DRIVE, LOCAL_GT_DIR)
    copy_dir_with_rsync_or_shutil(SPLIT_DIR_DRIVE, LOCAL_SPLIT_DIR)

    shutil.copy2(MAPPING_PATH_DRIVE, LOCAL_MAPPING_PATH)

print("Local features:", len(list(LOCAL_FEATURE_DIR.glob("*.npy"))))
print("Local groundTruth:", len(list(LOCAL_GT_DIR.glob("*.txt"))))
print("Local splits:", len(list(LOCAL_SPLIT_DIR.glob("*"))))
print("Local mapping:", LOCAL_MAPPING_PATH.exists())

assert local_dataset_is_complete(), "Local Breakfast copy is incomplete."


========== Copy dataset to local Colab disk ==========
Removing incomplete local copy: /content/breakfast_local
Running: rsync -ah --info=progress2 /content/drive/MyDrive/mmf_tas_lab_data/zenodo_ms_tcn_data/breakfast/features/ /content/breakfast_local/features/
Running: rsync -ah --info=progress2 /content/drive/MyDrive/mmf_tas_lab_data/zenodo_ms_tcn_data/breakfast/groundTruth/ /content/breakfast_local/groundTruth/
Running: rsync -ah --info=progress2 /content/drive/MyDrive/mmf_tas_lab_data/zenodo_ms_tcn_data/breakfast/splits/ /content/breakfast_local/splits/
Local features: 1712
Local groundTruth: 1712
Local splits: 9
Local mapping: True


## 5. Clone official MS-TCN repo

In [13]:
print("\n========== Clone official MS-TCN repo ==========")

if not REPO_ROOT.exists():
    subprocess.run(["git", "clone", "https://github.com/yabufarha/ms-tcn.git", str(REPO_ROOT)], check=True)
else:
    print("Repo already exists:", REPO_ROOT)

if WORK_ROOT.exists():
    shutil.rmtree(WORK_ROOT)

shutil.copytree(REPO_ROOT, WORK_ROOT)
print("Working copy:", WORK_ROOT)


========== Clone official MS-TCN repo ==========
Working copy: /content/ms-tcn-official-working


## 6. Patch official repo for Python 3

In [14]:
print("\n========== Patch official repo ==========")

def patch_file(path):
    path = Path(path)
    text = path.read_text()

    # Python 2 print statements -> Python 3 print(...)
    text = re.sub(r"^(\s*)print (?!\()(.+)$", r"\1print(\2)", text, flags=re.MULTILINE)

    # Python 2 dict iteration.
    text = text.replace(".iteritems()", ".items()")

    # Numpy aliases removed in newer numpy.
    text = text.replace("np.float", "float")
    text = text.replace("np.int", "int")

    # Python 3 map() returns an iterator; official batch_gen.py uses it more than once.
    text = text.replace(
        "length_of_sequences = map(len, batch_target)",
        "length_of_sequences = list(map(len, batch_target))",
    )

    # Python 3 dict view indexing fix in official predict().
    text = text.replace(
        "actions_dict.keys()[actions_dict.values().index(predicted[i].item())]",
        "list(actions_dict.keys())[list(actions_dict.values()).index(predicted[i].item())]",
    )

    path.write_text(text)


for py_name in ["model.py", "batch_gen.py", "main.py", "eval.py"]:
    py_path = WORK_ROOT / py_name
    if py_path.exists():
        patch_file(py_path)
        py_compile.compile(str(py_path), doraise=True)
        print("patched + compiled:", py_name)

sys.path.insert(0, str(WORK_ROOT))

from batch_gen import BatchGenerator
from model import MultiStageModel

print("Imported official BatchGenerator and MultiStageModel.")


========== Patch official repo ==========
patched + compiled: model.py
patched + compiled: batch_gen.py
patched + compiled: main.py
patched + compiled: eval.py
Imported official BatchGenerator and MultiStageModel.


## 7. Link local dataset into official repo layout

In [15]:
print("\n========== Link local dataset into official repo layout ==========")

official_data_root = WORK_ROOT / "data" / DATASET
official_data_root.mkdir(parents=True, exist_ok=True)

links = {
    official_data_root / "features": LOCAL_FEATURE_DIR,
    official_data_root / "groundTruth": LOCAL_GT_DIR,
    official_data_root / "splits": LOCAL_SPLIT_DIR,
}

for link_path, target_path in links.items():
    if link_path.exists() or link_path.is_symlink():
        if link_path.is_symlink():
            link_path.unlink()
        elif link_path.is_dir():
            shutil.rmtree(link_path)
        else:
            link_path.unlink()
    os.symlink(target_path, link_path)
    print(link_path, "->", link_path.resolve())

mapping_dst = official_data_root / "mapping.txt"
if mapping_dst.exists() or mapping_dst.is_symlink():
    mapping_dst.unlink()
os.symlink(LOCAL_MAPPING_PATH, mapping_dst)
print(mapping_dst, "->", mapping_dst.resolve())


========== Link local dataset into official repo layout ==========
/content/ms-tcn-official-working/data/breakfast/features -> /content/breakfast_local/features
/content/ms-tcn-official-working/data/breakfast/groundTruth -> /content/breakfast_local/groundTruth
/content/ms-tcn-official-working/data/breakfast/splits -> /content/breakfast_local/splits
/content/ms-tcn-official-working/data/breakfast/mapping.txt -> /content/breakfast_local/mapping.txt


## 8. Mapping and split sanity check

In [16]:
print("\n========== Mapping and split sanity check ==========")

def load_mapping(mapping_path):
    actions_dict = {}
    id_to_label = {}
    with open(mapping_path, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            class_id_str, label = line.split()
            class_id = int(class_id_str)
            actions_dict[label] = class_id
            id_to_label[class_id] = label
    return actions_dict, id_to_label

actions_dict, id_to_label = load_mapping(LOCAL_MAPPING_PATH)
num_classes = len(actions_dict)

train_split = LOCAL_SPLIT_DIR / f"train.split{SPLIT}.bundle"
test_split = LOCAL_SPLIT_DIR / f"test.split{SPLIT}.bundle"

train_entries = [x.strip() for x in train_split.read_text().splitlines() if x.strip()]
test_entries = [x.strip() for x in test_split.read_text().splitlines() if x.strip()]

sample_video = Path(train_entries[0]).stem
sample_feature = np.load(LOCAL_FEATURE_DIR / f"{sample_video}.npy")
sample_gt = [x.strip() for x in (LOCAL_GT_DIR / f"{sample_video}.txt").read_text().splitlines() if x.strip()]

print("num_classes:", num_classes)
print("train videos:", len(train_entries))
print("test videos:", len(test_entries))
print("sample_video:", sample_video)
print("sample_feature:", sample_feature.shape)
print("sample_gt frames:", len(sample_gt))

assert num_classes == 48
assert len(train_entries) == 1460
assert len(test_entries) == 252
assert sample_feature.ndim == 2
assert FEATURE_DIM in sample_feature.shape


========== Mapping and split sanity check ==========
num_classes: 48
train videos: 1460
test videos: 252
sample_video: P16_cam01_P16_cereals
sample_feature: (2048, 544)
sample_gt frames: 544


## 9. BatchGenerator smoke test

In [17]:
print("\n========== BatchGenerator smoke test ==========")

batch_gen_smoke = BatchGenerator(
    num_classes,
    actions_dict,
    str(LOCAL_GT_DIR) + "/",
    str(LOCAL_FEATURE_DIR) + "/",
    SAMPLE_RATE,
)
batch_gen_smoke.read_data(str(train_split))
batch_input, batch_target, mask = batch_gen_smoke.next_batch(1)

print("batch_input:", tuple(batch_input.shape))
print("batch_target:", tuple(batch_target.shape))
print("mask:", tuple(mask.shape))

assert batch_input.shape[1] == FEATURE_DIM
assert batch_target.shape[0] == 1
assert mask.shape[1] == num_classes


========== BatchGenerator smoke test ==========
batch_input: (1, 2048, 217)
batch_target: (1, 217)
mask: (1, 48, 217)


## 10. Training helpers with resume

In [18]:
print("\n========== Training helpers ==========")

history_path = LOG_DIR / "training_history.csv"
status_path = LOG_DIR / "pipeline_status.json"

def list_epoch_models(model_dir):
    model_dir = Path(model_dir)
    pairs = []
    for p in model_dir.glob("epoch-*.model"):
        m = re.search(r"epoch-(\d+)\.model$", p.name)
        if m:
            pairs.append((int(m.group(1)), p))
    return sorted(pairs)

def latest_epoch(model_dir):
    pairs = list_epoch_models(model_dir)
    if not pairs:
        return 0
    return pairs[-1][0]

def write_status(status):
    status["timestamp"] = datetime.utcnow().isoformat() + "Z"
    status_path.write_text(json.dumps(status, indent=2))
    print("Status saved:", status_path)

if FORCE_RETRAIN and MODEL_DIR.exists():
    print("Deleting old model dir:", MODEL_DIR)
    shutil.rmtree(MODEL_DIR)
    MODEL_DIR.mkdir(parents=True, exist_ok=True)

if FORCE_RETRAIN and RESULTS_DIR.exists():
    print("Deleting old results dir:", RESULTS_DIR)
    shutil.rmtree(RESULTS_DIR)
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)


========== Training helpers ==========


## 11. Train official MS-TCN

In [19]:
print("\n========== Train official MS-TCN ==========")

model = MultiStageModel(
    NUM_STAGES,
    NUM_LAYERS,
    NUM_F_MAPS,
    FEATURE_DIM,
    num_classes,
).to(device)

ce = torch.nn.CrossEntropyLoss(ignore_index=-100)
mse = torch.nn.MSELoss(reduction="none")
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

start_epoch = 1
last_epoch = latest_epoch(MODEL_DIR)

if last_epoch >= EPOCHS:
    print(f"Found existing epoch-{EPOCHS}.model. Skipping training.")
else:
    if last_epoch > 0:
        print(f"Resuming from epoch {last_epoch}.")
        model.load_state_dict(torch.load(MODEL_DIR / f"epoch-{last_epoch}.model", map_location=device))
        opt_path = MODEL_DIR / f"epoch-{last_epoch}.opt"
        if opt_path.exists():
            optimizer.load_state_dict(torch.load(opt_path, map_location=device))
            print("Loaded optimizer:", opt_path)
        start_epoch = last_epoch + 1
    else:
        print("No existing checkpoint. Starting from scratch.")

    batch_gen = BatchGenerator(
        num_classes,
        actions_dict,
        str(LOCAL_GT_DIR) + "/",
        str(LOCAL_FEATURE_DIR) + "/",
        SAMPLE_RATE,
    )
    batch_gen.read_data(str(train_split))

    if not history_path.exists():
        with open(history_path, "w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=["epoch", "epoch_loss", "train_acc", "elapsed_min"])
            writer.writeheader()

    training_start = time.time()

    for epoch in range(start_epoch, EPOCHS + 1):
        epoch_start = time.time()
        model.train()

        batch_gen.reset()

        epoch_loss = 0.0
        correct = 0.0
        total = 0.0
        num_batches = 0

        pbar = tqdm(total=len(batch_gen.list_of_examples), desc=f"Official MS-TCN epoch {epoch}/{EPOCHS}")

        while batch_gen.has_next():
            batch_input, batch_target, mask = batch_gen.next_batch(BATCH_SIZE)
            batch_input = batch_input.to(device)
            batch_target = batch_target.to(device)
            mask = mask.to(device)

            optimizer.zero_grad()

            predictions = model(batch_input, mask)

            loss = 0.0
            for p in predictions:
                loss += ce(
                    p.transpose(2, 1).contiguous().view(-1, num_classes),
                    batch_target.view(-1),
                )
                loss += 0.15 * torch.mean(
                    torch.clamp(
                        mse(
                            F.log_softmax(p[:, :, 1:], dim=1),
                            F.log_softmax(p.detach()[:, :, :-1], dim=1),
                        ),
                        min=0,
                        max=16,
                    )
                    * mask[:, :, 1:]
                )

            epoch_loss += float(loss.item())
            loss.backward()
            optimizer.step()

            _, predicted = torch.max(predictions[-1].data, 1)
            correct += float(((predicted == batch_target).float() * mask[:, 0, :]).sum().item())
            total += float(torch.sum(mask[:, 0, :]).item())

            num_batches += 1
            pbar.update(BATCH_SIZE)
            pbar.set_postfix(loss=f"{loss.item():.4f}", acc=f"{correct / max(total, 1):.4f}")

        pbar.close()

        train_acc = correct / max(total, 1.0)
        avg_loss = epoch_loss / max(len(batch_gen.list_of_examples), 1)
        elapsed_min = (time.time() - epoch_start) / 60.0

        model_path = MODEL_DIR / f"epoch-{epoch}.model"
        opt_path = MODEL_DIR / f"epoch-{epoch}.opt"

        torch.save(model.state_dict(), model_path)
        torch.save(optimizer.state_dict(), opt_path)

        row = {
            "epoch": epoch,
            "epoch_loss": avg_loss,
            "train_acc": train_acc,
            "elapsed_min": elapsed_min,
        }

        with open(history_path, "a", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=["epoch", "epoch_loss", "train_acc", "elapsed_min"])
            writer.writerow(row)

        print("[epoch %d]: epoch loss = %f, acc = %f, elapsed_min = %.2f" %
              (epoch, avg_loss, train_acc, elapsed_min))
        print("Saved:", model_path)

        write_status({
            "stage": "training",
            "completed_epoch": epoch,
            "target_epochs": EPOCHS,
            "train_acc": train_acc,
            "epoch_loss": avg_loss,
            "model_path": str(model_path),
            "artifact_root": str(ARTIFACT_ROOT),
        })

print("Training stage complete.")
assert (MODEL_DIR / f"epoch-{EPOCHS}.model").exists(), f"Missing final model epoch-{EPOCHS}.model"


========== Train official MS-TCN ==========
Resuming from epoch 16.
Loaded optimizer: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/official_mstcn_full_e30_oneshot/models/breakfast/split_1_official_full_e30/epoch-16.opt


Official MS-TCN epoch 17/30:   0%|          | 0/1460 [00:00<?, ?it/s]

[epoch 17]: epoch loss = 2.758819, acc = 0.817579, elapsed_min = 15.62
Saved: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/official_mstcn_full_e30_oneshot/models/breakfast/split_1_official_full_e30/epoch-17.model


/tmp/ipykernel_9902/2377734615.py:22: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  status["timestamp"] = datetime.utcnow().isoformat() + "Z"


Status saved: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/official_mstcn_full_e30_oneshot/logs/pipeline_status.json


...

[epoch 28]: epoch loss = 2.461684, acc = 0.863890, elapsed_min = 15.25
Saved: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/official_mstcn_full_e30_oneshot/models/breakfast/split_1_official_full_e30/epoch-28.model
Status saved: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/official_mstcn_full_e30_oneshot/logs/pipeline_status.json


Official MS-TCN epoch 29/30:   0%|          | 0/1460 [00:00<?, ?it/s]

[epoch 29]: epoch loss = 2.138487, acc = 0.884162, elapsed_min = 15.26
Saved: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/official_mstcn_full_e30_oneshot/models/breakfast/split_1_official_full_e30/epoch-29.model
Status saved: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/official_mstcn_full_e30_oneshot/logs/pipeline_status.json


Official MS-TCN epoch 30/30:   0%|          | 0/1460 [00:00<?, ?it/s]

[epoch 30]: epoch loss = 2.318431, acc = 0.879577, elapsed_min = 16.98
Saved: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/official_mstcn_full_e30_oneshot/models/breakfast/split_1_official_full_e30/epoch-30.model
Status saved: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/official_mstcn_full_e30_oneshot/logs/pipeline_status.json
Training stage complete.


## 12. Predict full test split

In [20]:
print("\n========== Predict full test split ==========")

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

model = MultiStageModel(
    NUM_STAGES,
    NUM_LAYERS,
    NUM_F_MAPS,
    FEATURE_DIM,
    num_classes,
).to(device)

final_model_path = MODEL_DIR / f"epoch-{EPOCHS}.model"
model.load_state_dict(torch.load(final_model_path, map_location=device))
model.eval()

id_to_label = {v: k for k, v in actions_dict.items()}

prediction_manifest = []
predict_start = time.time()

with torch.no_grad():
    for entry in tqdm(test_entries, desc="Predicting test videos"):
        video_file = Path(entry).name
        video_id = Path(entry).stem

        feature_path = LOCAL_FEATURE_DIR / f"{video_id}.npy"
        assert feature_path.exists(), feature_path

        features = np.load(feature_path).astype(np.float32)

        # Official Breakfast features are normally [D, T].
        if features.shape[0] != FEATURE_DIM and features.shape[1] == FEATURE_DIM:
            features = features.T

        features = features[:, ::SAMPLE_RATE]

        input_x = torch.tensor(features, dtype=torch.float32, device=device).unsqueeze(0)
        mask = torch.ones((1, num_classes, input_x.shape[-1]), dtype=torch.float32, device=device)

        predictions = model(input_x, mask)
        pred_ids = torch.argmax(predictions[-1], dim=1).squeeze(0).detach().cpu().numpy().tolist()

        pred_labels = [id_to_label[int(idx)] for idx in pred_ids]

        out_path = RESULTS_DIR / video_file
        out_path.write_text("### Frame level recognition: ###\n" + " ".join(pred_labels) + "\n")

        prediction_manifest.append({
            "video_id": video_id,
            "prediction_path": str(out_path),
            "num_pred_frames": len(pred_labels),
        })

manifest_path = RESULTS_DIR / "prediction_manifest.csv"
pd.DataFrame(prediction_manifest).to_csv(manifest_path, index=False)

print("Predictions saved:", RESULTS_DIR)
print("Prediction files:", len(list(RESULTS_DIR.glob("*.txt"))))
print("Manifest:", manifest_path)

assert len(list(RESULTS_DIR.glob("*.txt"))) == len(test_entries)


========== Predict full test split ==========


Predicting test videos:   0%|          | 0/252 [00:00<?, ?it/s]

Predictions saved: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/official_mstcn_full_e30_oneshot/results/breakfast/split_1_official_full_e30
Prediction files: 252
Manifest: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/official_mstcn_full_e30_oneshot/results/breakfast/split_1_official_full_e30/prediction_manifest.csv


## 13. Evaluation metrics

In [21]:
print("\n========== Evaluate official MS-TCN predictions ==========")

BACKGROUND_LABELS = {"background", "SIL", "sil"}

def read_gt_labels(path):
    return [x.strip() for x in Path(path).read_text().splitlines() if x.strip()]

def read_prediction_labels(path):
    lines = [x.strip() for x in Path(path).read_text().splitlines() if x.strip()]

    if len(lines) >= 2 and lines[0].startswith("###"):
        return lines[1].split()

    # fallback one-label-per-line
    return lines

def get_segments(frame_labels, background_labels=BACKGROUND_LABELS):
    labels = []
    starts = []
    ends = []

    last_label = None
    start = 0

    for i, label in enumerate(frame_labels):
        if last_label is None:
            last_label = label
            start = i
        elif label != last_label:
            if last_label not in background_labels:
                labels.append(last_label)
                starts.append(start)
                ends.append(i)
            last_label = label
            start = i

    if last_label is not None and last_label not in background_labels:
        labels.append(last_label)
        starts.append(start)
        ends.append(len(frame_labels))

    return labels, np.asarray(starts), np.asarray(ends)

def levenshtein_distance(pred_labels, gt_labels):
    m = len(pred_labels)
    n = len(gt_labels)

    dp = np.zeros((m + 1, n + 1), dtype=np.float32)
    dp[:, 0] = np.arange(m + 1)
    dp[0, :] = np.arange(n + 1)

    for i in range(1, m + 1):
        for j in range(1, n + 1):
            cost = 0 if pred_labels[i - 1] == gt_labels[j - 1] else 1
            dp[i, j] = min(
                dp[i - 1, j] + 1,
                dp[i, j - 1] + 1,
                dp[i - 1, j - 1] + cost,
            )

    return dp[m, n]

def edit_score(pred_frame_labels, gt_frame_labels):
    pred_labels, _, _ = get_segments(pred_frame_labels)
    gt_labels, _, _ = get_segments(gt_frame_labels)

    if len(pred_labels) == 0 and len(gt_labels) == 0:
        return 100.0

    denom = max(len(pred_labels), len(gt_labels))
    if denom == 0:
        return 0.0

    edit = levenshtein_distance(pred_labels, gt_labels)
    return (1.0 - edit / denom) * 100.0

def f_score_video(pred_frame_labels, gt_frame_labels, overlap):
    pred_labels, pred_starts, pred_ends = get_segments(pred_frame_labels)
    gt_labels, gt_starts, gt_ends = get_segments(gt_frame_labels)

    n_pred = len(pred_labels)
    n_gt = len(gt_labels)

    if n_pred == 0 and n_gt == 0:
        return 0, 0, 0
    if n_pred == 0:
        return 0, 0, n_gt
    if n_gt == 0:
        return 0, n_pred, 0

    hits = np.zeros(n_gt, dtype=np.float32)
    tp = 0
    fp = 0

    for j in range(n_pred):
        intersection = np.minimum(pred_ends[j], gt_ends) - np.maximum(pred_starts[j], gt_starts)
        union = np.maximum(pred_ends[j], gt_ends) - np.minimum(pred_starts[j], gt_starts)

        intersection = np.maximum(intersection, 0)
        iou = intersection / np.maximum(union, 1e-8)

        label_match = np.asarray([pred_labels[j] == gt_label for gt_label in gt_labels], dtype=bool)
        iou = iou * label_match

        idx = int(np.argmax(iou))
        if iou[idx] >= overlap and hits[idx] == 0:
            tp += 1
            hits[idx] = 1
        else:
            fp += 1

    fn = n_gt - int(hits.sum())
    return tp, fp, fn

total_correct = 0
total_frames = 0
edit_scores = []

f_counts = {
    0.10: {"tp": 0, "fp": 0, "fn": 0},
    0.25: {"tp": 0, "fp": 0, "fn": 0},
    0.50: {"tp": 0, "fp": 0, "fn": 0},
}

per_video_rows = []
missing_predictions = []
length_mismatches = []

for entry in tqdm(test_entries, desc="Evaluating"):
    video_file = Path(entry).name
    video_id = Path(entry).stem

    pred_path = RESULTS_DIR / video_file
    gt_path = LOCAL_GT_DIR / f"{video_id}.txt"

    if not pred_path.exists():
        missing_predictions.append(video_id)
        continue

    pred_labels = read_prediction_labels(pred_path)
    gt_labels = read_gt_labels(gt_path)

    if len(pred_labels) != len(gt_labels):
        length_mismatches.append({
            "video_id": video_id,
            "pred_len": len(pred_labels),
            "gt_len": len(gt_labels),
        })

    n = min(len(pred_labels), len(gt_labels))
    correct = int(np.sum(np.asarray(pred_labels[:n]) == np.asarray(gt_labels[:n])))

    total_correct += correct
    total_frames += n

    edit = edit_score(pred_labels, gt_labels)
    edit_scores.append(edit)

    row = {
        "video_id": video_id,
        "num_pred_frames": len(pred_labels),
        "num_gt_frames": len(gt_labels),
        "num_eval_frames": n,
        "length_difference": len(pred_labels) - len(gt_labels),
        "frame_accuracy": 100.0 * correct / max(n, 1),
        "edit": edit,
    }

    for overlap in f_counts:
        tp, fp, fn = f_score_video(pred_labels, gt_labels, overlap=overlap)
        f_counts[overlap]["tp"] += tp
        f_counts[overlap]["fp"] += fp
        f_counts[overlap]["fn"] += fn

    per_video_rows.append(row)

summary = {
    "model": "official_mstcn_visual_only",
    "dataset": DATASET,
    "split": SPLIT,
    "epochs": EPOCHS,
    "run_name": RUN_NAME,
    "num_prediction_files": len(list(RESULTS_DIR.glob("*.txt"))),
    "missing_predictions": len(missing_predictions),
    "length_mismatches": len(length_mismatches),
    "total_eval_frames": int(total_frames),
    "accuracy": 100.0 * total_correct / max(total_frames, 1),
    "edit": float(np.mean(edit_scores)) if edit_scores else 0.0,
}

for overlap, counts in f_counts.items():
    tp = counts["tp"]
    fp = counts["fp"]
    fn = counts["fn"]

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2.0 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

    suffix = int(overlap * 100)
    summary[f"precision@{suffix}"] = precision * 100.0
    summary[f"recall@{suffix}"] = recall * 100.0
    summary[f"f1@{suffix}"] = f1 * 100.0

df_summary = pd.DataFrame([summary])
df_per_video = pd.DataFrame(per_video_rows)
df_mismatches = pd.DataFrame(length_mismatches)

summary_csv = EVAL_DIR / "official_mstcn_tas_metrics_summary.csv"
per_video_csv = EVAL_DIR / "official_mstcn_tas_metrics_per_video.csv"
mismatches_csv = EVAL_DIR / "official_mstcn_length_mismatches.csv"
summary_json = EVAL_DIR / "official_mstcn_tas_metrics_summary.json"

df_summary.to_csv(summary_csv, index=False)
df_per_video.to_csv(per_video_csv, index=False)
df_mismatches.to_csv(mismatches_csv, index=False)
summary_json.write_text(json.dumps(summary, indent=2))

print("\n========== Final official MS-TCN metrics ==========")
display(df_summary[[
    "model",
    "accuracy",
    "edit",
    "f1@10",
    "f1@25",
    "f1@50",
    "missing_predictions",
    "length_mismatches",
    "total_eval_frames",
]])

print("Missing predictions:", len(missing_predictions))
print("Length mismatches:", len(length_mismatches))
print("Total eval frames:", total_frames)

if missing_predictions:
    print("First missing predictions:", missing_predictions[:10])

if length_mismatches:
    print("First length mismatches:")
    display(df_mismatches.head())

assert len(missing_predictions) == 0, "Some predictions are missing."
assert len(length_mismatches) == 0, "Some prediction lengths do not match ground truth."
assert total_frames > 0, "No frames were evaluated."


========== Evaluate official MS-TCN predictions ==========


Evaluating:   0%|          | 0/252 [00:00<?, ?it/s]


========== Final official MS-TCN metrics ==========


,model,accuracy,edit,f1@10,f1@25,f1@50,missing_predictions,length_mismatches,total_eval_frames
0,official_mstcn_visual_only,55.377882,44.974392,39.045093,34.801061,25.251989,0,0,505422


Missing predictions: 0
Length mismatches: 0
Total eval frames: 505422


## 14. Save Markdown report

In [22]:
print("\n========== Save report ==========")

report_path = EVAL_DIR / "official_mstcn_full_e30_report.md"

report_table = df_summary[["model", "accuracy", "edit", "f1@10", "f1@25", "f1@50"]].copy()
for col in ["accuracy", "edit", "f1@10", "f1@25", "f1@50"]:
    report_table[col] = report_table[col].map(lambda x: f"{x:.2f}")

report = f"""# Official MS-TCN Full Breakfast Baseline

Run: `{RUN_NAME}`

Dataset: `{DATASET}`

Split: `{SPLIT}`

Epochs: `{EPOCHS}`

Input: visual features only

Model source: official `yabufarha/ms-tcn` architecture, patched for Python 3 and modern Colab.

## Metrics

{report_table.to_markdown(index=False)}

## Validity checks

- Prediction files: {summary["num_prediction_files"]}
- Missing predictions: {summary["missing_predictions"]}
- Length mismatches: {summary["length_mismatches"]}
- Total evaluated frames: {summary["total_eval_frames"]}

## Artifact paths

- Artifact root: `{ARTIFACT_ROOT}`
- Model directory: `{MODEL_DIR}`
- Results directory: `{RESULTS_DIR}`
- Summary CSV: `{summary_csv}`
- Per-video CSV: `{per_video_csv}`
"""

report_path.write_text(report)

write_status({
    "stage": "completed",
    "target_epochs": EPOCHS,
    "artifact_root": str(ARTIFACT_ROOT),
    "model_dir": str(MODEL_DIR),
    "results_dir": str(RESULTS_DIR),
    "eval_dir": str(EVAL_DIR),
    "summary": summary,
})

print("Report saved:", report_path)
print(report)

print("\nDONE. Official MS-TCN full run completed and saved to Drive.")


========== Save report ==========
Status saved: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/official_mstcn_full_e30_oneshot/logs/pipeline_status.json
Report saved: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/official_mstcn_full_e30_oneshot/evaluation/official_mstcn_full_e30_report.md
# Official MS-TCN Full Breakfast Baseline

Run: `official_full_e30`

Dataset: `breakfast`

Split: `1`

Epochs: `30`

Input: visual features only

Model source: official `yabufarha/ms-tcn` architecture, patched for Python 3 and modern Colab.

## Metrics

| model                      |   accuracy |   edit |   f1@10 |   f1@25 |   f1@50 |
|:---------------------------|-----------:|-------:|--------:|--------:|--------:|
| official_mstcn_visual_only |      55.38 |  44.97 |   39.05 |    34.8 |   25.25 |

## Validity checks

- Prediction files: 252
- Missing predictions: 0
- Length mismatches: 0
- Total evaluated frames: 505422

## Artifact paths

- Artifact root: 

/tmp/ipykernel_9902/2377734615.py:22: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  status["timestamp"] = datetime.utcnow().isoformat() + "Z"
